# PySpark 3: types, functions and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-03-types-functions-and-nulls.ipynb)

A vendor file has amounts like `"1,204.50"`, dates like `"03/01/2026"`, missing regions, emails in mixed case,
and one amount of `"N/A"`. Before any join or report, every value needs the right type and a deliberate
answer for "what if it is missing or wrong?".

This lesson covers Spark's data types, casting under Spark 4's ANSI mode, how nulls behave, conditional logic,
and the built-in string, number, date and hashing functions you will use in almost every pipeline.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 2: DataFrame essentials](pyspark-02-dataframe-essentials.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

For everyday transformation work, focus first on column expressions, safe casts, null handling, and `when`. Use the string and date sections when turning raw source fields into join keys and business dates. Each example runs on a small input so you can predict what happens to a missing or invalid value before running the cell.

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
from decimal import Decimal

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Read a Spark column expression

Start with the built-in expressions you'll see throughout the lesson. `F.col("status")` refers to a field in the current row. `F.lit("paid")` supplies the same constant to every row. Functions such as `F.lower("status")` accept a column name as shorthand. In `F.coalesce("region", F.lit("unknown"))`, the first string is a column name and the second is a literal fallback.

Use `select` to choose and name the output columns. Use `withColumn` to add or replace a column while keeping the others. Both return a new DataFrame. Spark builds the execution plan as you chain expressions and evaluates it when you call an action such as `show()`. Keeping the logic in built-in column expressions gives Spark a plan it can optimize.

## Choose the right data type

| type | holds | use for |
| --- | --- | --- |
| `int`, `bigint` | 32- and 64-bit integers | ids, counts |
| `double` | binary floating point | measurements, scores, ratios |
| `decimal(p, s)` | exact numbers with `p` digits, `s` after the point | **money**, quantities that must add up |
| `string` | text | names, codes |
| `boolean` | true / false | flags |
| `date` | a calendar day | business dates, partitions |
| `timestamp` | an instant, shown in the session time zone | event times |
| `timestamp_ntz` | a wall-clock time with no time zone | store opening hours, local schedules |
| `array`, `struct`, `map` | nested values | covered in [lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb) |

A `double` stores numbers in binary, so most decimal fractions are approximations; sums of many of them drift.
A `decimal` stores them exactly, at the cost of a fixed number of digits.

**Where it matters**
- **Finance tables:** revenue reconciliations that must match to the cent.
- **Schemas for consumers:** downstream tools read your types; `bigint` vs `int` or `decimal` vs `double` is part of the contract.

**Watch out for**
- A decimal result can exceed its precision: Spark widens precision for arithmetic, up to 38 digits.
- Using `string` for everything postpones type errors to the consumer.

For example, `decimal(12,2)` allows ten digits before the decimal point and two after it. Choose precision and scale from the largest amount and smallest unit your source can represent. Keep identifiers like `"00042"` as strings when leading zeros matter. Use `printSchema()` after parsing or calculation to check the actual output type, especially when Spark has widened a decimal expression.

### Example

Add 0.1 and 0.2 as `double` and as `decimal`, and compare each result with 0.3.

In [ ]:
prices = spark.createDataFrame([(0.1, 0.2, Decimal("0.1"), Decimal("0.2"))], "a double, b double, c decimal(4,2), d decimal(4,2)")
result = prices.select(
    (F.col("a") + F.col("b")).alias("double_sum"),
    (F.col("a") + F.col("b") == 0.3).alias("double_equals_0_3"),
    (F.col("c") + F.col("d")).alias("decimal_sum"),
    (F.col("c") + F.col("d") == Decimal("0.3")).alias("decimal_equals_0_3"),
)
result.show()
result.printSchema()

**What to notice:** As doubles, 0.1 + 0.2 is `0.30000000000000004`, so it is not equal to 0.3: neither 0.1 nor 0.2 can be stored
exactly in binary. As decimals the sum is exactly `0.30`. The schema shows Spark widened the decimal result to
`decimal(5,2)` so the addition cannot overflow. Errors like this one, repeated over millions of rows, are why
revenue reconciliations fail on `double` columns.

## Cast types and handle bad values

Use `F.col("units_text").cast("int")` when invalid input should stop the pipeline. With ANSI mode enabled, as it is by default in Spark 4, malformed values and numeric overflow raise errors. The failure usually surfaces when an action evaluates the expression. Use `F.col("units_text").try_cast("int")` when malformed or out-of-range rows should be retained for review. Choose a supported conversion in both cases.

| Input | Function to reach for | Example |
| --- | --- | --- |
| Plain value with a known target type | `cast` or `try_cast` | `"4"` to an integer |
| Number with a formatting contract | `try_to_number` | `"1,204.50"` with `"9,999.99"` |
| Timestamp text with a known pattern | `try_to_timestamp` | `"31/01/2026 18:45"` with `"dd/MM/yyyy HH:mm"` |
| Ratio with a possible zero denominator | `try_divide` | A return rate for a group with zero orders |

Keep the raw and parsed columns long enough to tell an absent input from a failed conversion. If the raw value is present and parsing returns null, count or route the row as a reject. Decide how blanks should be treated before parsing. [Spark 4.0 PySpark try_cast reference](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.Column.try_cast.html)

### Example

Cast clean and dirty strings, see the error from a plain cast, and handle the same values with `try_` functions.

In [ ]:
raw = spark.createDataFrame(
    [("o1", "120.50", "4"), ("o2", "N/A", "0"), ("o3", "1,204.50", "2"), ("o4", None, "2")],
    "order_id string, amount_text string, units_text string",
)

spark.sparkContext.setLogLevel("OFF")  # the failure below is expected
try:
    raw.select(F.col("amount_text").cast("decimal(12,2)")).collect()
except Exception as exc:
    print("plain cast:", str(exc).splitlines()[0][:90])
finally:
    spark.sparkContext.setLogLevel("ERROR")

typed = raw.select(
    "order_id",
    "amount_text",
    F.col("amount_text").try_cast("decimal(12,2)").alias("try_cast"),
    F.try_to_number("amount_text", F.lit("9,999.99")).alias("try_to_number"),
    F.try_divide(F.lit(100), F.col("units_text").try_cast("int")).alias("per_unit"),
)
typed.withColumn(
    "plain_amount_rejected",
    F.col("amount_text").isNotNull() & F.col("try_cast").isNull(),
).orderBy("order_id").show()

**What to notice:** The plain cast failed on the whole job because of one value. `try_cast` returned null for both `N/A` and
`1,204.50`, since the comma is not part of a plain number. `try_to_number` with the format `9,999.99` parsed
`1,204.50` correctly and still returned null for `N/A`. `try_divide` returned null for the division by zero
instead of failing.

Rows `o2` and `o3` both fail the plain-decimal contract. The formatted parser accepts `o3`, so the reject definition depends on which source contract applies. Row `o4` is a missing input and the reject flag stays false.

## Handle nulls

Spark SQL uses three-valued logic. Ordinary comparisons such as `region == "North"` and `region != "North"` return null when the region is missing. A filter keeps only true rows. Negating a predicate with `~` leaves unknown values unknown. Combine Spark predicates with `&` and `|`, and parenthesize each comparison because Spark evaluates a column expression for every row.

| `region` | `region != "North"` | `~region.eqNullSafe("North")` |
| --- | --- | --- |
| North | false | false |
| South | true | true |
| null | null | true |

Use `isNull()` to find missing values. Use `eqNullSafe` when comparing two nullable values, such as before and after snapshots. Two nulls compare equal, and a null paired with a known value compares unequal. In a combined flag, `false & null` is false and `true & null` is null. [Spark SQL null semantics](https://spark.apache.org/docs/4.0.1/sql-ref-null-semantics.html)

| Need | Expression |
| --- | --- |
| Take the first available phone or default | `F.coalesce("mobile", "home", F.lit("unavailable"))` |
| Treat an empty value as missing | `F.nullif(F.trim("email"), F.lit(""))` |
| Fill selected fields according to a business rule | `df.fillna({"region": "unknown"})` |
| Drop rows with any missing required key | `df.dropna(subset=["customer_id", "order_id"])` |
| Drop rows where every candidate field is missing | `df.dropna(how="all", subset=["mobile", "home"])` |

`count("*")` counts rows. `count("region")` counts known regions. `sum` and `avg` skip nulls and return null when all inputs are null. For example, an order group with all amounts missing still has orders. Count rows to measure volume and count known amounts to measure completeness.

### Example

Compare with and without null safety, fill and drop nulls, turn an empty string into null, and count with and without nulls.

In [ ]:
contacts = spark.createDataFrame(
    [(1, "North", "a@x.com"), (2, None, ""), (3, "South", None), (4, None, "d@x.com")],
    "id int, region string, email string",
)
contacts.select(
    "id", "region",
    (F.col("region") == F.lit(None)).alias("equals_null"),
    F.col("region").eqNullSafe(F.lit(None)).alias("null_safe"),
    F.coalesce("region", F.lit("unknown")).alias("region_or_default"),
    F.nullif("email", F.lit("")).alias("email_clean"),
).show()

print("dropna(subset=['region']):", contacts.dropna(subset=["region"]).count(), "rows")
contacts.fillna({"region": "unknown", "email": "none"}).show()
contacts.agg(F.count("*").alias("rows"), F.count("region").alias("with_region")).show()

region = F.col("region")
contacts.select(
    "id", "region",
    (region != "North").alias("ordinary_comparison"),
    (~region.eqNullSafe("North")).alias("include_unknowns"),
).orderBy("id").show()

**What to notice:** `region == null` is null for every row, even where region is null, so it can never select anything; only
`eqNullSafe` returns true for the missing regions. `nullif` turned customer 2's empty email into a real null.
`count("*")` counts 4 rows but `count("region")` only the 2 known regions.

A filter on `ordinary_comparison` keeps only the South row. A filter on `include_unknowns` keeps that row and both rows with a missing region. Choose the predicate based on whether the report should include customers whose region is unknown.

## Write conditional logic

Use `F.when(condition, value).when(...).otherwise(default)` for a per-row classification. Spark picks the first true condition, so put a narrow rule before a broader rule that would also match. A null condition does not match, and omitting `otherwise` gives unmatched rows null. If a missing amount should be labeled `"unknown"`, handle it in an explicit first branch. Keep result branches in compatible types.

Use `isin("paid", "shipped")` to test membership in a small fixed list. For rules maintained in a separate table, a lookup join lets the data carry the mapping. Use `greatest` or `least` when choosing among several columns in the same row. Both skip nulls and return null if every candidate is null.

A conditional expression can feed an aggregate. Use `count_if` to count true predicates, or use `sum(when(...).otherwise(...))` when each qualifying row contributes an amount. Decide whether other rows should contribute zero or represent missing data.

### Example

Assign size tiers with `when`, and show what happens to rows that match no branch when `otherwise` is missing.

In [ ]:
orders.select(
    "order_id", "status", "amount",
    F.when(F.col("amount") >= 1000, "enterprise")
     .when(F.col("amount") >= 100, "mid_market")
     .otherwise("small").alias("tier"),
    F.when(F.col("status").isin("paid", "shipped"), True).alias("revenue_flag"),
).orderBy("order_id").show()

**What to notice:** Only order 9 is `enterprise`, because the `>= 1000` branch is tested first. `revenue_flag` has no `otherwise`, so
placed and returned orders get null instead of false; a later `where(~revenue_flag)` would miss them.

In [ ]:
is_paid = F.col("status") == "paid"
orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"),
    F.count_if(is_paid).alias("paid_orders"),
    F.sum(F.when(is_paid, F.col("amount")).otherwise(D("0.00"))).alias("paid_amount"),
).orderBy("customer_id").show()

Customer 4 has two orders and zero paid orders. The explicit zero branch gives that group a paid amount of `0.00`. Leaving off `otherwise` would give the group an all-null input to `sum` and a null paid amount.

## Work with strings

Start with the simplest operation that matches the source format. For a case-insensitive email key, `lower(trim(email))` handles case and outer spaces. Preserve the raw value for troubleshooting, and apply the same normalization on both join inputs. Use `nullif` if a blank result should count as missing.

| Task | Functions | Behavior to check |
| --- | --- | --- |
| Clean case and outer spaces | `trim`, `lower`, `upper`, `initcap` | Inner spaces remain after `trim` |
| Combine fields | `concat`, `concat_ws` | `concat` propagates a null input. `concat_ws` skips null fields but keeps empty strings |
| Read fixed positions or format an ID | `substring`, `length`, `lpad` | Substring positions start at 1. `lpad` truncates strings longer than the requested width |
| Split a delimited field | `split` | The separator is a regex, so a literal pipe needs `r"\|"` |
| Extract or replace a pattern | `regexp_extract`, `regexp_replace` | Group 0 is the full match. An unmatched extraction on a non-null input gives an empty string |

For a code such as `UK-LON-0042`, anchoring the regex with `^` and `$` validates the entire code. Wrap the extraction in `nullif(..., F.lit(""))` if a nonmatch should be missing. If an ID is part of a key, check its length before padding so a long ID cannot be silently shortened.

### Example

Normalize emails, build a display name, pad an id, extract parts with regular expressions, and split a code.

In [ ]:
people = spark.createDataFrame(
    [("  Ada@Example.COM ", "ada", "lovelace", 7, "UK-LON-0042"),
     ("grace@example.org", "grace", None, 42, "US-NYC-0007"),
     ("lin@example.org", "lin", "chen", 108, "unassigned")],
    "email string, first string, last string, id int, location string",
)
people.select(
    F.lower(F.trim("email")).alias("email"),
    F.concat(F.initcap("first"), F.lit(" "), F.initcap("last")).alias("concat"),
    F.concat_ws(" ", F.initcap("first"), F.initcap("last")).alias("concat_ws"),
    F.lpad(F.col("id").cast("string"), 5, "0").alias("padded_id"),
    F.nullif(
        F.regexp_extract("location", r"^([A-Z]{2})-([A-Z]{3})-(\d+)$", 2), F.lit("")
    ).alias("city"),
    F.regexp_replace("location", r"\d", "#").alias("masked"),
    F.split("location", "-").alias("parts"),
).show(truncate=False)

**What to notice:** Grace has no last name: `concat` returned null for her, while `concat_ws` skipped the null and returned
`Grace`. `regexp_extract` with group 2 pulled out the city code, and `split` returned an array you can index
later. `lpad` made every id five characters wide.

The `unassigned` value has no city match. `nullif` turns the empty extraction into null, which can be counted or routed as a missing code.

## Round and compute numbers

`round(col, n)` rounds half up (2.5 becomes 3); `bround` rounds half to even (2.5 becomes 2), which can reduce
bias in large sums. `ceil`, `floor`, `abs` and `pow` do what their names say. Keep money in `decimal` and
round only for display or at a defined business step, not at every intermediate calculation.

**Where it matters**
- **Financial reporting rules,** which often specify the rounding mode.
- **Percentages and ratios** computed from integer counts.

**Watch out for**
- Integer division does not truncate in Spark: `7 / 2` is `3.5`. Use `F.floor` to round down or SQL `div` to truncate the fractional part.
- Rounding before summing gives a different total than summing before rounding.

### Example

Compare `round` and `bround` on halves, and divide integers.

In [ ]:
halves = spark.createDataFrame([(Decimal("0.5"),), (Decimal("1.5"),), (Decimal("2.5"),)], "value decimal(3,1)")
halves.select(
    "value", F.round("value").alias("round"), F.bround("value").alias("bround"),
    F.ceil("value").alias("ceil"), F.floor("value").alias("floor"),
).show()
spark.range(1).select((F.lit(7) / 2).alias("7 / 2"), F.expr("7 div 2").alias("7 div 2")).show()

**What to notice:** On the positive halves shown, `round` gives 1, 2, 3 and `bround` gives 0, 2, 2. Half-up rounds exact ties away from zero. Half-even goes to the nearest even value and can reduce systematic bias when many values land on ties. Use the mode specified by the business rule. Ordinary division of the two integers gives 3.5, while `div` gives 3. For negative values, `floor` rounds toward negative infinity and `div` truncates toward zero.

## Work with dates and timestamps

Parse source text into a typed value before filtering or grouping. Use `date` for a calendar day and `timestamp` for an event instant. Use `timestamp_ntz` when the source represents a wall-clock reading without enough information to place it on a global timeline.

| Need | Functions | Detail |
| --- | --- | --- |
| Parse a date or timestamp | `to_date`, `to_timestamp`, `try_to_timestamp` | Use the source pattern and a tolerant parser when bad input should become null |
| Report elapsed calendar days | `datediff(end, start)` | Argument order determines the sign |
| Move by days or months | `date_add`, `add_months`, `last_day` | Adding a month to January 31 lands on February's last valid day |
| Group by calendar period | `trunc`, `date_trunc` | `trunc(date, "month")` returns a date. `date_trunc("month", ts)` returns a timestamp |
| Display a date or derive a label | `date_format`, `year`, `month`, `dayofweek` | `date_format` returns a string. `dayofweek` gives Sunday 1 and Saturday 7 |
| Read Unix epoch milliseconds | `timestamp_millis` | Check the input unit, since epoch seconds need `timestamp_seconds` |

Pattern letters are case-sensitive. In `"yyyy-MM-dd HH:mm"`, `MM` is month and `mm` is minute. If two formats are accepted, try each explicitly in a documented order and keep invalid rows visible. For bounded reporting periods, apply both boundaries, such as July 1 inclusive and January 1 exclusive for the second half of the year.

The session time zone controls how Spark displays timestamps and derives dates from them. This notebook pins the session to UTC. For a Chicago business date, it shifts the instant to Chicago wall-clock time with `from_utc_timestamp`, then takes the date in the UTC-pinned session. Keep the original timestamp for event comparisons and use the derived date for reporting. A named zone such as `America/Chicago` accounts for daylight saving time. For source strings with an offset, parse the offset into the instant before deriving the business date. [Spark 4.0 PySpark from_utc_timestamp reference](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_utc_timestamp.html)

### Example

Parse European-style dates, derive calendar fields, and convert order 12 from UTC to Chicago time.

In [ ]:
deliveries = spark.createDataFrame([("03/01/2026", "2026-01-31 18:45")], "day_text string, ts_text string")
deliveries.select(
    F.to_date("day_text", "dd/MM/yyyy").alias("day"),
    F.to_timestamp("ts_text", "yyyy-MM-dd HH:mm").alias("ts"),
    F.date_format(F.to_date("day_text", "dd/MM/yyyy"), "EEEE d MMMM yyyy").alias("pretty"),
    F.last_day(F.to_date("day_text", "dd/MM/yyyy")).alias("month_end"),
    F.add_months(F.to_date("day_text", "dd/MM/yyyy"), 1).alias("next_month"),
).show(truncate=False)

orders.where("order_id in (11, 12)").select(
    "order_id", "order_ts",
    F.to_date("order_ts").alias("utc_date"),
    F.from_utc_timestamp("order_ts", "America/Chicago").alias("chicago_time"),
    F.to_date(F.from_utc_timestamp("order_ts", "America/Chicago")).alias("chicago_date"),
    F.dayofweek("order_ts").alias("day_of_week"),
    F.datediff(F.lit("2026-01-31").cast("date"), F.to_date("order_ts")).alias("days_to_month_end"),
).show()

**What to notice:** `03/01/2026` became 3 January because the format says day first. Order 12 was placed at 00:30 UTC on 14 January,
which is still 13 January in Chicago, so the two business dates differ. `dayofweek` counts from Sunday = 1.

In [ ]:
date_inputs = spark.createDataFrame(
    [("2026-01-31",), ("01/31/2026",), ("2026-02-30",), (None,)],
    "date_text string",
)
parsed_date = F.to_date(F.coalesce(
    F.try_to_timestamp("date_text", F.lit("yyyy-MM-dd")),
    F.try_to_timestamp("date_text", F.lit("MM/dd/yyyy")),
))
date_inputs.select("date_text", parsed_date.alias("business_date")).show()

Both valid inputs yield January 31. February 30 and the missing input yield null. As with numeric casts, keep the source column until you have counted invalid supplied values separately from missing inputs.

## Generate ids and hashes

- `sha2(col, 256)` and `md5` produce stable hashes of a value: the same input always gives the same output.
  `xxhash64` is a fast 64-bit hash. For a composite key, define a stable encoding that preserves field boundaries and null positions before hashing.
- `monotonically_increasing_id()` gives each row a unique 64-bit number, but **not** consecutive or stable:
  the values encode the partition, and change if the data is repartitioned.
- `F.expr("uuid()")` generates a random identifier per row in Spark 4.0. Treat it as non-reproducible across runs.

**Where it matters**
- **Surrogate keys and change detection:** hash the business key, or hash all attributes to spot changed rows.
- **Matching records** within a controlled pipeline when the same canonical input should produce the same key.

**Watch out for**
- Never use `monotonically_increasing_id` or `uuid` as a key you need to reproduce on a rerun.
- Plain concatenation turns both `('ab', 'c')` and `('a', 'bc')` into the same text. A separator can also be ambiguous if it appears in the data.

A plain hash of an email can be guessed by hashing candidate addresses. Treat it as sensitive. Also check ambiguous encodings before hashing a composite key. For example, `concat_ws` with a pipe produces the same text for `("A", null, "B")` and `("A", "B", null)`.

### Example

Hash emails and a row's attributes, and see what `monotonically_increasing_id` produces across partitions.

In [ ]:
customers.select(
    "customer_id",
    F.sha2(F.lower("name"), 256).substr(1, 16).alias("name_sha256"),
    F.xxhash64("customer_id", "name", "region").alias("row_hash"),
).show(3, truncate=False)

spark.range(6, numPartitions=3).select("id", F.monotonically_increasing_id().alias("mono_id")).show()

**What to notice:** The hashes are deterministic: rerun the cell and they stay the same. The `mono_id` values jump between
partitions (0, 1, then 8589934592, ...) because the upper bits hold the partition number. They are unique but
not consecutive, and a different partitioning would give different numbers.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why store money as `decimal` rather than `double`?

**A.** A `double` stores binary approximations, so sums drift (ten times 0.1 is 0.9999999999999999). A `decimal` is exact.

**Q2.** In Spark 4, what does casting `"N/A"` to `int` do, and how do you get a null instead?

**A.** ANSI mode is on, so the cast raises `CAST_INVALID_INPUT` and the job fails. `try_cast` returns null.

**Q3.** Why does `region == None` never select anything?

**A.** Any comparison with null gives null, and a filter keeps only true rows. Use `isNull()` or `eqNullSafe`.

**Q4.** What is the difference between `count('*')` and `count('region')`?

**A.** `count('*')` counts rows; `count('region')` counts non-null values of `region`.

**Q5.** Why might `concat` return null where `concat_ws` does not?

**A.** `concat` returns null if any input is null; `concat_ws` skips nulls.

**Q6.** Can you use `monotonically_increasing_id` as a primary key?

**A.** Only within one run. The values depend on partitioning, are not consecutive, and change on a rerun. Hash a business key instead.

## Key takeaways

- Money is `decimal`; measurements can be `double`.
- Spark 4 casts fail loudly; use `try_` functions where bad values are expected, then count the nulls.
- Nulls fail comparisons and are skipped by aggregates: handle them on purpose.
- Order `when` branches from the most specific; always decide what `otherwise` should be.
- Give date formats explicitly and pin the session time zone.
- Hash business keys for reproducible ids.

**Next:** Practise this module in [PySpark drills 3: types, functions and nulls](pyspark-drills-03-types-functions-and-nulls.ipynb), then continue with [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb).